# 01 — Data & point-in-time leak checks

Build the panel (synthetic by default: planted betas, known truth), then run the leak diagnostics **before any evaluation**. The rule (roadmap §0): point-in-time correctness in both directions — no forward leakage, no stale information.

In [ ]:
# Path shim: make the repo root importable when running from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 140)


In [ ]:
import yaml
from src.data.synthetic import make_synthetic_panel
from src.utils.stats import rank_normalize_cross_section

cfg = yaml.safe_load(open(ROOT / "configs/config.yaml"))
panel, factors, meta = make_synthetic_panel(cfg, seed=cfg["run"]["seed"])
signal_cols = list(meta.index)
panel = rank_normalize_cross_section(panel, signal_cols)
print(panel["date"].nunique(), "months x", panel["ticker"].nunique(), "names")
meta


## Predictive vs contemporaneous

The only tradeable number is the IC against **forward** returns. The contemporaneous column is shown to make the comparison explicit and auditable.

In [ ]:
from src.data.panel import leak_report
leak_report(panel, signal_cols)

## The deliberate lookahead demonstration

A feature contaminated with the return it claims to predict produces an *absurd* IC (~0.4 when honest single signals live near 0.02–0.06). If a real feature ever looks like this, audit the timestamps.

In [ ]:
from src.data.panel import demonstrate_lookahead
demonstrate_lookahead(panel, seed=cfg['run']['seed'])

## Staleness profile ('Anomaly Time', JF 2024)

IC as the signal goes stale. Fresh information should dominate; a steep drop means formation timing is itself a research variable.

In [ ]:
from src.data.panel import staleness_experiment
stale = staleness_experiment(panel, signal_cols, max_lag=6)
stale.pivot(index='staleness_months', columns='signal', values='IC').round(4)